# 06 · Lo que sí funciona

Indicadores en los que Uruguay queda **primero** entre los 12 países de la comparación (toda Sudamérica, Costa Rica y México), con América Latina y la OCDE como referencia cuando hay dato.

**Fuente única:** API del Banco Mundial.

| Código | Indicador | Base |
|---|---|---|
| GOV_WGI_CC.SC | Control de la corrupción (puntaje 0–100) | Worldwide Governance Indicators (WGI), `source=3` |
| GOV_WGI_RL.SC | Estado de derecho (0–100) | WGI |
| GOV_WGI_VA.SC | Voz y rendición de cuentas (0–100) | WGI |
| GOV_WGI_PV.SC | Estabilidad política y ausencia de violencia (0–100) | WGI |
| EG.ELC.RNWX.ZS | Electricidad renovable sin hidroeléctrica (% del total) | World Development Indicators (WDI) |
| IT.NET.BBND.P2 | Suscripciones de banda ancha fija cada 100 habitantes | WDI (dato original: UIT) |

**Ojo:** los puntajes del WGI combinan más de 30 encuestas y evaluaciones de expertos. Miden percepciones, no hechos contados uno por uno, y tienen márgenes de error. Se usa el puntaje 0–100 porque es más fácil de leer que la escala original (−2,5 a 2,5).


## 1. Descarga (bronze)

In [ ]:
import os, json, urllib.request

RAW = "/Volumes/workspace/pobreza/raw/banco_mundial"
os.makedirs(RAW, exist_ok=True)

PAISES = ["URY", "ARG", "BOL", "BRA", "CHL", "COL", "ECU", "PRY", "PER", "VEN", "CRI", "MEX", "LCN", "OED"]
# (código en la API, source): source=3 es la base WGI, source=2 es WDI
INDICADORES = [("GOV_WGI_CC.SC", 3), ("GOV_WGI_RL.SC", 3), ("GOV_WGI_VA.SC", 3), ("GOV_WGI_PV.SC", 3),
               ("EG.ELC.RNWX.ZS", 2), ("IT.NET.BBND.P2", 2)]

for ind, src in INDICADORES:
    url = (f"https://api.worldbank.org/v2/country/{';'.join(PAISES)}/indicator/{ind}"
           f"?format=json&source={src}&date=2012:2025&per_page=5000")
    with urllib.request.urlopen(url) as r:
        datos = json.loads(r.read())
    with open(f"{RAW}/{ind}.json", "w") as f:
        json.dump(datos[1], f)
    print(ind, len(datos[1]), "filas")

In [ ]:
from pyspark.sql import functions as F

filas = []
for ind, _ in INDICADORES:
    for r in json.load(open(f"{RAW}/{ind}.json")):
        filas.append((ind.replace("GOV_WGI_", ""), r["countryiso3code"], r["date"],
                      None if r["value"] is None else float(r["value"])))

bronze = spark.createDataFrame(filas, "indicador string, iso3 string, anio string, valor double")
bronze.write.mode("overwrite").option("overwriteSchema", True).saveAsTable("workspace.pobreza.fortalezas_bronze")
print(bronze.count(), "filas")

## 2. Silver

- Se sacan los años sin dato (la OCDE no tiene puntajes WGI como agregado).
- Nombres en español, año entero, valor con 2 decimales.

In [ ]:
NOMBRES = {"URY": "Uruguay", "ARG": "Argentina", "BRA": "Brasil", "CHL": "Chile", "PRY": "Paraguay",
           "CRI": "Costa Rica", "MEX": "México", "BOL": "Bolivia", "COL": "Colombia",
           "ECU": "Ecuador", "PER": "Perú", "VEN": "Venezuela", "LCN": "América Latina y el Caribe", "OED": "OCDE"}
mapa = F.create_map([F.lit(x) for kv in NOMBRES.items() for x in kv])

silver = (spark.table("workspace.pobreza.fortalezas_bronze")
    .filter(F.col("valor").isNotNull())
    .select("indicador", "iso3", mapa[F.col("iso3")].alias("pais"),
            F.col("anio").cast("int").alias("anio"), F.round("valor", 2).alias("valor")))
silver.write.mode("overwrite").option("overwriteSchema", True).saveAsTable("workspace.pobreza.fortalezas_silver")

## 3. Gold + export

In [0]:
import os

def carpeta_data():
    ruta = os.path.dirname(dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get())
    if not ruta.startswith("/Workspace"):
        ruta = "/Workspace" + ruta
    if os.path.basename(ruta) == "notebooks":   # si estoy en la carpeta notebooks...
        ruta = os.path.dirname(ruta)            # ...subo un nivel, a la raíz del repo
    os.makedirs(f"{ruta}/data", exist_ok=True)
    return f"{ruta}/data"

def guardar_gold(df, nombre):
    df.write.mode("overwrite").option("overwriteSchema", True).saveAsTable(f"workspace.pobreza.{nombre}_gold")
    df.toPandas().to_json(f"{carpeta_data()}/{nombre}.json", orient="records", force_ascii=False, indent=1)
    print("OK:", nombre)

In [ ]:
gold = spark.table("workspace.pobreza.fortalezas_silver").orderBy("indicador", "iso3", "anio")
guardar_gold(gold, "fortalezas_paises")

# Puesto de Uruguay entre los 12 países en el último dato
from pyspark.sql.window import Window
ult = (gold.filter(F.col("iso3").isin(PAISES[:12]))
       .withColumn("r", F.row_number().over(Window.partitionBy("indicador", "iso3").orderBy(F.desc("anio"))))
       .filter("r = 1").drop("r")
       .withColumn("puesto", F.rank().over(Window.partitionBy("indicador").orderBy(F.desc("valor")))))
display(ult.filter("iso3 = 'URY'"))

## Hallazgos (último dato)

- Uruguay es **1º de 12** en los seis indicadores.
- Control de la corrupción: **79** (Chile 70, Costa Rica 62). Estado de derecho: **74**. Voz y rendición de cuentas: **80**. Estabilidad política: **88** (2025).
- Electricidad de viento, sol y biomasa: de **13%** en 2012 a **51%** en 2021; el promedio de la OCDE es 17%.
- Banda ancha fija: **33 cada 100 habitantes** (2024), cerca del promedio de la OCDE (35 en 2022).